In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.data.market_loader import MarketLoader
from src.data.synthetic_sofr_builder import build_term_sofr_curve

from src.term_structure.curve_merger import merge_curves
from src.term_structure.bootstrapping import (
    build_coupon_structure,
    bootstrap_dfs_from_sofr,
    bootstrap_dfs_from_treasury
)
from src.term_structure.curve_interpolator import log_linear_curve_interpolator
from src.term_structure.curve_builder import build_zero_curve


In [2]:
### Curve Builder
# downloading market curves
loader = MarketLoader()
curves = loader.loader_pipeline()

# building synthetic sofr curve from ON rates and futures
sofr_curve = build_term_sofr_curve(curves = curves)

# bootstrapping discount factors from sofr curve
df_sofr = bootstrap_dfs_from_sofr(
    sofr_curve = sofr_curve
)

# extracting treasury curve 
treasury_curve = curves['treasury']

# bootstrapping discount factors from treasury curve
df_treasury = bootstrap_dfs_from_treasury(
    treasury_curve = treasury_curve,
    short_dfs = df_sofr
)

# concatenating short-end and long-end into a single curve -> full curve
df_full_curve = merge_curves(
    short_curve = df_sofr,
    long_curve = df_treasury
)

# constructing semi-annual coupon structure
coupon_structure = build_coupon_structure(
    max_year = 30,
    freq = 2
)

# log-linear discount curve interpolation
df_loglinear_interp = log_linear_curve_interpolator(
    df_curve = df_full_curve,
    _target_times = coupon_structure
)

# building zero curve from log-linear discount curve
zero_curve_full = build_zero_curve(
    df_curve = df_loglinear_interp
)

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..


In [3]:

key_tenors =  [1, 2, 3, 5, 7, 10, 20, 30]
zero_curve = zero_curve_full[key_tenors]
zero_curve

,1.0,2.0,3.0,5.0,7.0,10.0,20.0,30.0
2018-04-03,0.020802,0.022699,0.024494,0.025930,0.027055,0.027900,0.029826,0.030468
2018-04-04,0.020604,0.022701,0.024552,0.026032,0.027096,0.027894,0.029913,0.030586
2018-04-05,0.020603,0.022901,0.024807,0.026333,0.027458,0.028301,0.030319,0.030991
2018-04-06,0.020505,0.022602,0.024339,0.025730,0.026856,0.027700,0.029715,0.030386
2018-04-09,0.020702,0.022799,0.024537,0.025928,0.026993,0.027792,0.029811,0.030484
...,...,...,...,...,...,...,...,...
2026-04-15,0.036671,0.037267,0.038056,0.038687,0.041116,0.042937,0.048664,0.050573
2026-04-16,0.036574,0.037469,0.038199,0.038784,0.041340,0.043257,0.049090,0.051034
2026-04-17,0.036080,0.036777,0.037508,0.038094,0.040710,0.042673,0.048585,0.050556
2026-04-20,0.036176,0.036874,0.037663,0.038294,0.040784,0.042651,0.048573,0.050546


In [4]:
import numpy as np

delta_yields = zero_curve.diff().dropna() # type: ignore
mean_changes = delta_yields.mean()
centered_delta_yields = delta_yields - mean_changes
cov_matrix = centered_delta_yields.cov()
eigen_vals, eigen_vecs = np.linalg.eigh(cov_matrix)


In [5]:
eigen_vals

array([-1.90427687e-22, -4.11831925e-23,  2.28288557e-22,  6.54027291e-09,
        2.07547454e-08,  5.23754702e-08,  2.96062657e-07,  1.93625541e-06])

In [6]:
eigen_vecs

array([[-2.21652912e-15, -6.24640388e-16, -2.27478879e-15,
         2.88107176e-02, -4.06755750e-01, -7.75865484e-01,
         4.20733679e-01, -2.33956459e-01],
       [-4.49187381e-02, -3.18895676e-01, -1.75545696e-01,
        -2.81971478e-01,  6.72764813e-01, -9.47369218e-03,
         4.53905951e-01, -3.56694256e-01],
       [ 1.01067161e-01,  7.17515271e-01,  3.94977815e-01,
         1.91508359e-01,  1.43902665e-01,  1.96548657e-01,
         2.75057341e-01, -3.83768739e-01],
       [ 1.19099606e-01, -5.15677784e-01,  6.58548135e-02,
         5.70292229e-01, -2.79187053e-01,  3.61366537e-01,
         1.31978454e-01, -4.05428326e-01],
       [-4.08912067e-01,  2.73135774e-01, -6.65669510e-01,
        -1.42255179e-01, -2.91522880e-01,  2.30382917e-01,
        -6.44858109e-02, -3.90659750e-01],
       [ 5.43972109e-02, -1.77479944e-01,  4.59108210e-01,
        -6.76665735e-01, -3.00774750e-01,  1.32145201e-01,
        -2.11834009e-01, -3.79583318e-01],
       [ 7.17067309e-01,  8.560943

In [7]:
eigen_vecs[:, 7]

array([-0.23395646, -0.35669426, -0.38376874, -0.40542833, -0.39065975,
       -0.37958332, -0.3316791 , -0.31571102])

In [8]:
eigen_vecs[:, 6]

array([ 0.42073368,  0.45390595,  0.27505734,  0.13197845, -0.06448581,
       -0.21183401, -0.44656553, -0.52480936])

In [9]:
idx = np.argsort(eigen_vals)[::-1]
eigen_vals[idx]

array([ 1.93625541e-06,  2.96062657e-07,  5.23754702e-08,  2.07547454e-08,
        6.54027291e-09,  2.28288557e-22, -4.11831925e-23, -1.90427687e-22])

In [10]:
eigen_vecs[:, idx]

array([[-2.33956459e-01,  4.20733679e-01, -7.75865484e-01,
        -4.06755750e-01,  2.88107176e-02, -2.27478879e-15,
        -6.24640388e-16, -2.21652912e-15],
       [-3.56694256e-01,  4.53905951e-01, -9.47369218e-03,
         6.72764813e-01, -2.81971478e-01, -1.75545696e-01,
        -3.18895676e-01, -4.49187381e-02],
       [-3.83768739e-01,  2.75057341e-01,  1.96548657e-01,
         1.43902665e-01,  1.91508359e-01,  3.94977815e-01,
         7.17515271e-01,  1.01067161e-01],
       [-4.05428326e-01,  1.31978454e-01,  3.61366537e-01,
        -2.79187053e-01,  5.70292229e-01,  6.58548135e-02,
        -5.15677784e-01,  1.19099606e-01],
       [-3.90659750e-01, -6.44858109e-02,  2.30382917e-01,
        -2.91522880e-01, -1.42255179e-01, -6.65669510e-01,
         2.73135774e-01, -4.08912067e-01],
       [-3.79583318e-01, -2.11834009e-01,  1.32145201e-01,
        -3.00774750e-01, -6.76665735e-01,  4.59108210e-01,
        -1.77479944e-01,  5.43972109e-02],
       [-3.31679098e-01, -4.465655

In [11]:
eigen_vals = eigen_vals[idx]
eigen_vals[:3]

array([1.93625541e-06, 2.96062657e-07, 5.23754702e-08])

In [12]:
eigen_vecs_temp = eigen_vecs[:, idx]
eigen_vecs_temp[:, :3]

array([[-0.23395646,  0.42073368, -0.77586548],
       [-0.35669426,  0.45390595, -0.00947369],
       [-0.38376874,  0.27505734,  0.19654866],
       [-0.40542833,  0.13197845,  0.36136654],
       [-0.39065975, -0.06448581,  0.23038292],
       [-0.37958332, -0.21183401,  0.1321452 ],
       [-0.3316791 , -0.44656553, -0.21696585],
       [-0.31571102, -0.52480936, -0.33333621]])